# 🚀 SimpleAI — TinyGPT 加法实验 T4 并行加速版 (32 路并行 + 预存数据 + 零容错评测)

> **核心定位**：在 Google Colab 上实现加法 Transformer 实验的**一键自动化训练（Train）**、**40 道测试题零容错严格判分（Verify）** 与 **学术级机制归因总结报告生成**。  
> **极简直连架构**：
> - **代码与配置源**：直接极速拉取 Hugging Face 公开仓库 (`https://huggingface.co/Hana-ame/additive-rand-transformer`)，公开免密，直连 100MB/s+，彻底规避任何 GitHub 跨站点或子模块协议报错；
> - **产物持久化**：训练产物（Checkpoint `.pt`、40 题评测大表、学术汇报 Markdown）直接自动同步归档到 **Google Drive**。

---

## ⚡ 极速操作说明 (How To Run)
1. 在顶部菜单栏点击 **代码执行程序 (Runtime) -> 更改运行时类型 (Change runtime type)**，确认硬件加速器选择 **GPU (T4 / A100 / L4)**。
2. 菜单栏直接点击 **代码执行程序 (Runtime) -> 全部运行 (Run All)**（或按快捷键 `Ctrl+F9`）。
3. 运行中会提示授权挂载 Google Drive，运行完毕后全部 Checkpoint 与评测大表将自动保存在您的 Google Drive `MyDrive/SimpleAI_Experiments/` 目录中！


### 步骤 1：GPU 硬件检测与挂载 Google Drive (Hardware & Google Drive Setup)


In [ ]:
import os, sys, time, json, shutil
import torch
from google.colab import drive

print("=" * 65)
print("🚀 SimpleAI 实验执行环境检测")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU 检测成功: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("⚠️ 未检测到 GPU，将在 CPU 模式下运行（建议切换至 GPU 运行时以提高训练速度）")
print("=" * 65)

# 挂载 Google Drive，完成的 artifact 自动保存到此处
try:
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/SimpleAI_Experiments'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"✅ Google Drive 挂载成功！全部产物将自动归档至: {DRIVE_DIR}")
except Exception as e:
    DRIVE_DIR = None
    print(f"⚠️ Google Drive 挂载跳过: {e}，产物将保存在 Colab 本地运行区")


### 步骤 2：直连 Hugging Face 拉取代码与配置 (Direct HF Clone & Import)
> 直接拉取公开托管在 Hugging Face 的代码仓库，无需任何 Token / 密码，0 门槛秒级就绪。


In [ ]:
# 1. 安装核心运行依赖
!pip install -q torch openpyxl huggingface_hub pandas matplotlib tabulate

import os, sys

# 2. 直连 Hugging Face 公开仓库
os.chdir('/content')
WORKSPACE = "/content/additive-rand-transformer"

if not os.path.exists(WORKSPACE):
    print("🌐 正在从 Hugging Face 极速克隆仓库与 440 项实验配置...")
    os.system(f"git clone https://huggingface.co/Hana-ame/additive-rand-transformer {WORKSPACE}")
else:
    print("🔄 仓库已存在，拉取 Hugging Face 最新代码...")
    os.system(f"git -C {WORKSPACE} pull || true")

os.chdir(WORKSPACE)

# 3. 配置 Python sys.path
if WORKSPACE not in sys.path:
    sys.path.insert(0, WORKSPACE)

from additive_rand_transformer.model import TinyGPT, TinyGPTConfig, VOCAB_SIZE, TOK_TO_ID
from additive_rand_transformer.data import BOS, EOS, PLUS, MINUS, EQ, SP, ANS, ANS_END, _int_to_tokens, extract_answer
print(f"✅ 环境准备完毕！当前工作区: {WORKSPACE}")
print(f"✅ 词表大小: {VOCAB_SIZE} Tokens (已启用 <ANS> ... </ANS> 零容错闭合严格判定)")


### 步骤 3：40 道基准测试题零容错严格判分引擎 (引擎已封装在 Python 模块)

> 评测引擎（40 题集 + 零容错判分 + 32 词表 `<ANS>…</ANS>` 严格闭合校验）已从 notebook
> 抽离到代码仓库模块 `additive_rand_transformer/parallel_train.py`，本单元格仅做导入。

In [ ]:
# 40 题零容错判分引擎已抽离到 Python 模块 (notebook 仅作启动器)
from additive_rand_transformer.parallel_train import (
    TEST_40_QUESTIONS,
    verify_model_40_questions,
    run_parallel_batch,
)
print(f"✅ 40 题零容错严格判分评测引擎已从模块加载 ({len(TEST_40_QUESTIONS)} 题)")

### 步骤 4：🚀 T4 并行加速训练 + 自动评测 (一行启动，逻辑在 Python 模块)

> 全部训练/并行/评测逻辑都封装在 `additive_rand_transformer/parallel_train.py`：
> - **32 路单进程多线程并行**（模型仅 926K 参数，数据预生成驻留内存，T4 满载）
> - 每个实验独立模型+优化器，输出到 `runs/par/<cfg>/`
> 下方仅需配置 3 个参数即启动。

In [ ]:
# ==============================================================================
# 🎯 运行模式选择 (具体逻辑见 additive_rand_transformer/parallel_train.py)
# ==============================================================================
RUN_MODE = "VOCAB32_PAIR_417_424"  # 可选: "FRONTIER_197_204", "VOCAB32_PAIR_417_424", "RUN_ALL_UNRUN"
MAX_EXPERIMENTS = 32               # 本次批量运行的上限个数
PARALLEL = 32                      # 🟢 并行度: 数据和模型都极小, 单进程多线程即可
# ==============================================================================

# 一行启动: 预生成数据 -> 32 路并行训练 -> 40 题零容错评测 -> 返回 reports
all_experiment_reports = run_parallel_batch(
    run_mode=RUN_MODE,
    max_experiments=MAX_EXPERIMENTS,
    parallel=PARALLEL,
    configs_dir=os.path.join(WORKSPACE, "configs"),
    device=("cuda" if torch.cuda.is_available() else "cpu"),
)

### 步骤 5：展示 40 题得分明细矩阵 (View 40-Question Detailed Results)


In [ ]:
from tabulate import tabulate

for rep in all_experiment_reports:
    print()
    print("#" * 80)
    print(f"📋 实验报告: {rep['title']} ({rep['config']})")
    print(f"得分: {rep['score']}/40 ({rep['score']/40*100:.1f}%) | 词表: {rep['vocab_size']} | 耗时: {rep['duration']:.1f}s")
    print("#" * 80)
    
    table_data = []
    for d in rep['details']:
        status_icon = "🟢 PASS (1分)" if d['pass'] else "🔴 FAIL (0分)"
        pred_display = str(d['pred']) if d['pred'] is not None else "格式错/空"
        table_data.append([d['qid'], d['expr'], d['target'], pred_display, status_icon, d['desc']])
        
    print(tabulate(table_data, headers=["题号", "算式", "真值", "模型输出", "判定结果", "题型特点"], tablefmt="grid"))


### 步骤 6：生成学术级《实验结论与机制归因汇总报告》(`EXPERIMENT_CONCLUSIONS_REPORT.md`)
报告严格执行学术规范，包含假说检验与因果机制分析！


In [ ]:
report_path = "EXPERIMENT_CONCLUSIONS_REPORT.md"

# 动态统计本次 Colab 实际运行的真实指标（严禁任何预填或假数据）
report_lines = [
    "# 📑 SimpleAI 本次 Colab 真实运行评测报告 (Actual Run Report)",
    "",
    f"> **评测时间**：{time.strftime('%Y-%m-%d %H:%M:%S')}",
    f"> **执行硬件**：{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}",
    "> **数据真实性说明**：本报告所有指标均由本次 Colab 运行 40 道测试题零容错判定引擎现场评测得出，绝无任何人工预设或硬编码数据。",
    "",
    "---",
    "",
    "## 🏆 一、本次实际运行得分总览",
    "",
    "| 实验配置编号 | 实验标题 | 词表 | 40题总得分 | 得分率 | 耗时 (s) | 状态 |",
    "|---|---|:---:|:---:|:---:|:---:|:---:|"
]

if not all_experiment_reports:
    report_lines.append("| — | 本次未运行任何实验 | — | 0/40 | 0.0% | 0s | 未执行 |")
else:
    for rep in all_experiment_reports:
        status = "🟢 优秀 (>=80%)" if rep["score"] >= 32 else ("🟡 及格 (>=60%)" if rep["score"] >= 24 else "🔴 待攻关 (<60%)")
        report_lines.append(f"| `{rep['config']}` | {rep['title']} | {rep['vocab_size']} | **{rep['score']}/40** | {rep['score']/40*100:.1f}% | {rep['duration']:.1f}s | {status} |")

report_lines.extend([
    "",
    "---",
    "",
    "## 🔬 二、各实验 40 题真实能力分项实测明细",
    ""
])

for rep in all_experiment_reports:
    details = rep["details"]
    
    # 真实统计分项得分
    add1_pass = sum(1 for d in details if d["qid"] in ["Q01","Q02","Q03","Q04","Q05"] and d["pass"])
    add2_pass = sum(1 for d in details if d["qid"] in ["Q06","Q07","Q08","Q09","Q10"] and d["pass"])
    add3_pass = sum(1 for d in details if d["qid"] in ["Q11","Q12","Q13","Q14","Q15"] and d["pass"])
    add4_pass = sum(1 for d in details if d["qid"] in ["Q16","Q17","Q18","Q19","Q20"] and d["pass"])
    sub1_pass = sum(1 for d in details if d["qid"] in ["Q21","Q22","Q23","Q24","Q25"] and d["pass"])
    sub2_pass = sum(1 for d in details if d["qid"] in ["Q26","Q27","Q28","Q29","Q30"] and d["pass"])
    sub3_pass = sum(1 for d in details if d["qid"] in ["Q31","Q32","Q33","Q34","Q35"] and d["pass"])
    sub4_pass = sum(1 for d in details if d["qid"] in ["Q36","Q37","Q38","Q39","Q40"] and d["pass"])
    
    # 关键机制题检测
    q14_pass = next((d["pass"] for d in details if d["qid"] == "Q14"), False) # 999+1
    q18_pass = next((d["pass"] for d in details if d["qid"] == "Q18"), False) # 9999+1
    q36_pass = next((d["pass"] for d in details if d["qid"] == "Q36"), False) # 1000-1
    
    report_lines.extend([
        f"### 实验: `{rep['config']}` — {rep['title']}",
        f"* **实测总得分**: **{rep['score']} / 40** ({rep['score']/40*100:.1f}%)",
        f"* **加法分项掌握率**:",
        f"  * 1位加法 (Add1): {add1_pass}/5 ({add1_pass*20}%)",
        f"  * 2位加法 (Add2): {add2_pass}/5 ({add2_pass*20}%)",
        f"  * 3位加法 (Add3): {add3_pass}/5 ({add3_pass*20}%)",
        f"  * 4位加法 (Add4): {add4_pass}/5 ({add4_pass*20}%)",
        f"* **减法分项掌握率**:",
        f"  * 1位减法 (Sub1): {sub1_pass}/5 ({sub1_pass*20}%)",
        f"  * 2位减法 (Sub2): {sub2_pass}/5 ({sub2_pass*20}%)",
        f"  * 3位减法 (Sub3): {sub3_pass}/5 ({sub3_pass*20}%)",
        f"  * 4位减法 (Sub4): {sub4_pass}/5 ({sub4_pass*20}%)",
        f"* **极端进位/退位雪崩探针真值**:",
        f"  * `Q14 (999+1)`: {'🟢 PASS' if q14_pass else '🔴 FAIL'}",
        f"  * `Q18 (9999+1)`: {'🟢 PASS' if q18_pass else '🔴 FAIL'}",
        f"  * `Q36 (1000-1)`: {'🟢 PASS' if q36_pass else '🔴 FAIL'}",
        f"* **机制归因初判**: {'【连续进位掌握良好】多位进位雪崩测试均已攻破' if (q14_pass and q18_pass) else '【进位链路仍存在瓶颈】高位或雪崩进位题仍有失分，注意力在长程反向寻址上仍有衰减'}",
        ""
    ])

report_lines.extend([
    "---",
    "*本报告由 Google Colab 现场实测生成 | 严禁伪造数据*"
])

with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines) + "\n")

print(f"✅ 基于实测数据的真实评测报告已生成: {report_path}")


### 步骤 7：完成的 Artifact 全部自动归档至 Google Drive (Save to GDrive)
将全部实验产物（40题得分 CSV 大表、Checkpoints 权重、学术归因报告）自动同步至 Google Drive！


In [ ]:
import pandas as pd
from google.colab import files

print("=" * 65)
print("💾 正在将全部实验产物 (Artifacts) 归档持久化...")
print("=" * 65)

# 1. 构造 40 题全量得分明细表格
flat_rows = []
for rep in all_experiment_reports:
    for d in rep["details"]:
        flat_rows.append({
            "实验配置": rep["config"],
            "实验标题": rep["title"],
            "总得分": rep["score"],
            "词表": rep["vocab_size"],
            "耗时(s)": f"{rep['duration']:.1f}",
            "题号": d["qid"],
            "算式": d["expr"],
            "标准真值": d["target"],
            "模型预测": d["pred"] if d["pred"] is not None else "None(未闭合/格式错)",
            "判定结果": "PASS" if d["pass"] else "FAIL",
            "题型归类": d["desc"]
        })

df_scorecard = pd.DataFrame(flat_rows)
scorecard_csv = "evaluation_40_questions_scorecard.csv"
df_scorecard.to_csv(scorecard_csv, index=False, encoding="utf-8-sig")
print(f"✓ 40 题逐题得分明细大表已生成: {scorecard_csv}")

# 2. 如果挂载了 Google Drive，自动全量同步
if DRIVE_DIR:
    shutil.copy("EXPERIMENT_CONCLUSIONS_REPORT.md", os.path.join(DRIVE_DIR, "EXPERIMENT_CONCLUSIONS_REPORT.md"))
    shutil.copy(scorecard_csv, os.path.join(DRIVE_DIR, scorecard_csv))
    print(f"✓ 报告与数据表已同步至 Google Drive: {DRIVE_DIR}")
    
    if os.path.exists("runs"):
        runs_dest = os.path.join(DRIVE_DIR, "runs")
        os.makedirs(runs_dest, exist_ok=True)
        os.system(f"cp -ru runs/* {runs_dest}/ 2>/dev/null || true")
        print(f"✓ Checkpoints 与 Runs 训练日志已备份至: {runs_dest}")
    print()
    print(f"🎉 恭喜！全部实验产物已安全归档至 Google Drive 目录: {DRIVE_DIR}")
else:
    print("ℹ️ 未挂载 Drive，触发浏览器直接下载...")
    files.download(scorecard_csv)
    files.download("EXPERIMENT_CONCLUSIONS_REPORT.md")


### 步骤 8：【独立一键重评/补救引擎】直接扫描 runs/ 下全部已存 Checkpoint 评测并同步 Drive
> **用途**：当训练还在进行或已经跑完时，无需重新训练（0 算力浪费），随时运行本单元格即可直接提取 `runs/` 下所有已保存的 `checkpoint_final.pt` 权重文件，以正确的标签兼容逻辑完成 40 题实测，并将大表和模型自动备份到 Google Drive。

In [ ]:
import os, glob, time, shutil, json
import torch
import pandas as pd
from tabulate import tabulate
from additive_rand_transformer.model import TinyGPT, TinyGPTConfig
from additive_rand_transformer.data import extract_answer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔍 启动独立评测引擎，计算硬件: {device}")

# 1. 扫描 runs/ 目录下所有已保存的 checkpoint_final.pt（按生成时间排序）
all_ckpts = sorted(glob.glob("runs/**/checkpoint_final.pt", recursive=True), key=os.path.getmtime)
print(f"📦 共检索到 {len(all_ckpts)} 个已保存的 Checkpoint 文件:")
for p in all_ckpts:
    print(f"  - {p}")

rescued_reports = []

for ckpt_path in all_ckpts:
    run_dir = os.path.dirname(ckpt_path)
    run_name = os.path.basename(run_dir)
    print()
    print("=" * 70)
    print(f"🚀 正在评测已存权重: {ckpt_path}")
    print("=" * 70)
    
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    raw_cfg = ck.get("config", {})
    model_cfg = TinyGPTConfig(**{k: v for k, v in raw_cfg.items() if hasattr(TinyGPTConfig, k)})
    eval_model = TinyGPT(model_cfg).to(device)
    eval_model.load_state_dict(ck["model"])
    
    # 智能自适应标签：根据模型实际词表与配置，判断是否需要 <ANS> 标签
    use_ans_tags = raw_cfg.get("use_ans_tags", False) or (raw_cfg.get("vocab_size") == 32)
    answer_order = raw_cfg.get("answer_order", "msd")
    
    score, test_details = verify_model_40_questions(eval_model, device=device,
                                                    answer_order=answer_order,
                                                    require_tags=use_ans_tags)
    
    print(f"✅ 评测完毕! 40题实测得分: {score}/40 ({score/40*100:.1f}%) | 标签模式: {use_ans_tags} | 答案顺序: {answer_order}")
    
    rescued_reports.append({
        "ckpt_path": ckpt_path,
        "run_name": run_name,
        "score": score,
        "vocab_size": raw_cfg.get("vocab_size", 16),
        "answer_order": answer_order,
        "use_ans_tags": use_ans_tags,
        "details": test_details
    })

# 2. 构造 40 题逐题明细大表
flat_rows = []
for rep in rescued_reports:
    for d in rep["details"]:
        flat_rows.append({
            "运行目录": rep["run_name"],
            "权重路径": rep["ckpt_path"],
            "40题总得分": rep["score"],
            "得分率": f"{rep['score']/40*100:.1f}%",
            "题号": d["qid"],
            "算式": d["expr"],
            "标准真值": d["target"],
            "模型预测": d["pred"] if d["pred"] is not None else "None(空/未闭合)",
            "判定": "PASS" if d["pass"] else "FAIL",
            "题型特点": d["desc"]
        })

df_rescued = pd.DataFrame(flat_rows)
scorecard_path = "evaluation_40_questions_scorecard.csv"
df_rescued.to_csv(scorecard_path, index=False, encoding="utf-8-sig")
print()
print(f"📊 40 题逐题得分明细大表已更新: {scorecard_path}")

# 3. 自动同步至 Google Drive
drive_dest = '/content/drive/MyDrive/simpleAI_workspace'
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(drive_dest, exist_ok=True)
    shutil.copy(scorecard_path, os.path.join(drive_dest, scorecard_path))
    os.system(f"cp -ru runs/ {drive_dest}/runs/ 2>/dev/null || true")
    print(f"🎉 全部已存模型权重与 40 题实测得分明细已成功备份至 Google Drive: {drive_dest}")
else:
    print("ℹ️ 未挂载 Drive，产物保存在 Colab 当前目录。")
